[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_04_ML_I_II_Recap.ipynb)

# NB Warm-up 04 — Build, fit, predict, select

**STAT3009 · Recommender Systems**  
**Suggested time:** 18–20 minutes

> **ML I → ML II:** specify a rating rule, put it in a sklearn-style estimator, then choose its hyperparameter with cross-validation.

**Route:** recall (2 min) → model and hand calculation (4 min) → `fit`/`predict` (6 min) → `GridSearchCV` (6 min) → exit ticket (2 min). The rating data are included below; no download is needed.


## 1. Quick recall: which operation does what?

Decide first; then open the answers.

1. Is a constructor setting such as `strength=2.0` chosen before or learned during `fit`?
2. Can `predict(X_new)` use the unknown ratings of `X_new`?
3. In a CV fold, should user and item means be calculated from all ratings or only that fold's fitting rows?
4. Why is sklearn's RMSE score **negative**?

<details>
<summary><b>Recall answers</b></summary>

1. Chosen before `fit`: it is a **hyperparameter**.
2. No. `predict` receives only the user–item pairs.
3. Only fitting rows. Validation ratings must stay unseen until scoring.
4. sklearn maximizes scores; `neg_root_mean_squared_error` makes smaller RMSE correspond to a larger score. Negate it again when reporting RMSE.

</details>


## 2. A small recommendation problem

Each row of `X_ratings` is `[user_id, item_id]`; the matching entry of `y_ratings` is its rating. IDs are already nonnegative integers. Users and items have different numbers of observed ratings. Can we let the amount of evidence on **both sides** affect a prediction?


In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.utils.validation import check_is_fitted

n_users, n_items = 5, 4  # full ID vocabulary, set before any fit or CV fold

X_ratings = np.array([
    [0, 0], [0, 1],
    [1, 0], [1, 1], [1, 2], [1, 3],
    [2, 0], [2, 2], [2, 3],
    [3, 1], [3, 2], [3, 3],
    [4, 0], [4, 1], [4, 2], [4, 3],
])
y_ratings = np.array([
    5., 4.,
    1., 2., 2., 1.,
    4., 4., 5.,
    3., 4., 3.,
    2., 3., 3., 2.,
])

print('X shape:', X_ratings.shape)
print('y shape:', y_ratings.shape)
print('ID vocabulary:', n_users, 'users and', n_items, 'items')
print('global mean:', y_ratings.mean())


## 3. The four questions before the class

**1 · Model.** For a pair $(u,i)$, blend the user, item, and global training means:

$$\widehat r_{ui}=\frac{n_u\bar r_u+n_i\bar r_i+\lambda\mu}{n_u+n_i+\lambda}.$$

Here $n_u$ and $n_i$ count **training** ratings for the user and item. Their weights are $n_u/D$ and $n_i/D$, while the global mean gets $\lambda/D$, where $D=n_u+n_i+\lambda$. The weights add to 1. An unseen user or item has count zero; if both are unseen, the prediction is $\mu$.

**2 · Hyperparameter.** `strength` is $\lambda>0$, chosen before `fit`. Increasing it pulls predictions more toward the global mean. `n_users` and `n_items` are fixed ID-vocabulary sizes, not settings selected by CV.

**3 · Fitting / selection criterion.** `fit` calculates sample means and counts; this rule does **not** solve a joint loss-minimization problem. We compare candidate values of $\lambda$ by **validation RMSE** using `GridSearchCV`.

**4 · Learned parameters.** `global_mean_`, `user_means_`, `user_counts_`, `item_means_`, and `item_counts_` come only from the ratings passed to `fit`.

**Hand calculation:** User 0 has mean $4.5$ from **2** ratings. Item 2 has mean $3.25$ from **4** ratings, and $\mu=3$. With $\lambda=2$, what are the three weights and the prediction for `(0, 2)`? What changes for a **new user** rating item 2?


### One-line NumPy practice

For `(0, 2)`, put the user count, item count, and global strength into a NumPy array. Divide by their total to obtain the three weights. Write the expression below, then compare with the solution.


In [ ]:
strength = 2.0
counts = np.array([2, 4, strength])  # user count, item count, global strength

# Your turn: divide counts by their total and print weights.
# weights = ...
# print(weights)


<details>
<summary><b>NumPy answer</b></summary>

`weights = counts / counts.sum()` gives `[0.25, 0.5, 0.25]`. The user, item, and global contributions add to one.

</details>


## 4. Build a sklearn-style recommender estimator

Read the class in two passes. In `fit`, find where the global mean and both sets of means and counts are learned. In `predict`, find where the two counts and `strength` combine into the denominator. The arrays use integer IDs as lookup positions, as in ML I. An ID absent from this fit has count zero and a global-mean fallback.

The constructor stores the full ID-vocabulary sizes and `strength`. That allows `GridSearchCV` to make a **fresh estimator** for each candidate and fold.


In [ ]:
class CountAwareBlendRS(RegressorMixin, BaseEstimator):
    def __init__(self, n_users, n_items, strength=2.0):
        self.n_users = n_users
        self.n_items = n_items
        self.strength = strength

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=int), np.asarray(y, dtype=float)
        users, items = X[:, 0], X[:, 1]

        self.global_mean_ = y.mean()
        self.user_means_ = np.full(self.n_users, self.global_mean_)
        self.user_counts_ = np.zeros(self.n_users, dtype=int)
        self.item_means_ = np.full(self.n_items, self.global_mean_)
        self.item_counts_ = np.zeros(self.n_items, dtype=int)

        for user in set(users):
            ratings = y[users == user]
            self.user_means_[user] = ratings.mean()
            self.user_counts_[user] = len(ratings)
        for item in set(items):
            ratings = y[items == item]
            self.item_means_[item] = ratings.mean()
            self.item_counts_[item] = len(ratings)
        return self

    def predict(self, X):
        check_is_fitted(self, ['global_mean_', 'user_means_',
                               'user_counts_', 'item_means_', 'item_counts_'])
        X = np.asarray(X, dtype=int)
        users, items = X[:, 0], X[:, 1]

        user_mean = np.full(len(X), self.global_mean_)
        item_mean = np.full(len(X), self.global_mean_)
        user_count = np.zeros(len(X), dtype=int)
        item_count = np.zeros(len(X), dtype=int)
        known_user = (users >= 0) & (users < len(self.user_counts_))
        known_item = (items >= 0) & (items < len(self.item_means_))
        user_mean[known_user] = self.user_means_[users[known_user]]
        user_count[known_user] = self.user_counts_[users[known_user]]
        item_mean[known_item] = self.item_means_[items[known_item]]
        item_count[known_item] = self.item_counts_[items[known_item]]

        total = user_count + item_count + self.strength
        return (user_count * user_mean + item_count * item_mean
                + self.strength * self.global_mean_) / total


### Your turn: fit and predict

Make a `CountAwareBlendRS(n_users=n_users, n_items=n_items, strength=2.0)`, fit it on `X_ratings, y_ratings`, and predict the four pairs below. Decide first which ones need a fallback. The first prediction should match your hand calculation.


In [ ]:
X_new = np.array([
    [0, 2],   # known user, known item; this pair was not in training
    [99, 2],  # new user
    [0, 99],  # new item
    [99, 99], # both new
])

# Your turn: create model, call fit, then call predict on X_new.
# model = ...
# model.fit(..., ...)
# predictions = ...
# print(predictions)


### Solution


In [ ]:
model = CountAwareBlendRS(n_users=n_users, n_items=n_items, strength=2.0)
model.fit(X_ratings, y_ratings)
predictions = model.predict(X_new)

print('user 0 count:', model.user_counts_[0])
print('user 0 mean:', model.user_means_[0])
print('item 2 count:', model.item_counts_[2])
print('item 2 mean:', model.item_means_[2])
print('predictions:', np.round(predictions, 3))


The predictions, rounded to three decimals, are `[3.500, 3.167, 3.750, 3.000]`. For `(0, 2)`, the weights are $(2,4,2)/8=(0.25,0.5,0.25)$, giving $0.25(4.5)+0.5(3.25)+0.25(3)=3.5$. A new user has $n_u=0$, so item 2 and the global mean receive weights $4/6$ and $2/6$. When both IDs are new, only the global mean remains.


## 5. Choose `strength` with `GridSearchCV`

Now compare three **positive** settings: `0.5`, `2.0`, and `8.0`. The shuffled three-fold split here asks about **held-out rating pairs**; it is not an evaluation of entirely new users. The full ID-vocabulary sizes stay fixed, while each fold calculates its means and counts from **its own fitting rows**.

Use `neg_root_mean_squared_error` for scoring, then change the sign back when you report CV RMSE. `refit=True` fits the winning setting on all 16 available ratings after CV. This tiny dataset is for practicing the workflow, not for claiming which setting will work best in general.


In [ ]:
# Your turn: complete the same GridSearchCV pattern used in ML II.
# cv = KFold(n_splits=3, shuffle=True, random_state=42)
# search = GridSearchCV(
#     estimator=CountAwareBlendRS(n_users=n_users, n_items=n_items),
#     param_grid={'strength': ...},
#     scoring='...',
#     cv=cv,
#     refit=True,
# )
# search.fit(X_ratings, y_ratings)


### Solution and score reading


In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=CountAwareBlendRS(n_users=n_users, n_items=n_items),
    param_grid={'strength': [0.5, 2.0, 8.0]},
    scoring='neg_root_mean_squared_error',
    cv=cv,
    refit=True,
)
search.fit(X_ratings, y_ratings)

for strength, neg_rmse in zip(search.cv_results_['param_strength'],
                             search.cv_results_['mean_test_score']):
    print(f'strength={strength}: CV RMSE={-neg_rmse:.3f}')

print('selected strength:', search.best_params_['strength'])
print(f'best CV RMSE: {-search.best_score_:.3f}')
print('refitted global mean:', search.best_estimator_.global_mean_)
print('prediction for (0, 2):', search.best_estimator_.predict([[0, 2]]))


## Exit ticket

1. Which value did `GridSearchCV` choose, and where do you read it? Which arrays did the selected estimator **learn** when `refit=True`?
2. Why would precomputing user or item means from all 16 ratings **before** CV leak information into validation?
3. If the task were to predict ratings for **entirely new users**, would this pair-level CV split answer the right question?

<details>
<summary><b>After discussion</b></summary>

Read the chosen setting from `best_params_['strength']`. The refitted estimator is `best_estimator_`; its global, user, and item means and both count arrays use all 16 training ratings. Precomputing means or counts before CV would let validation ratings affect the rule used to predict themselves. Entirely new users require holding out whole users rather than random pairs.

</details>
